## Modeling - Decision Tree Regressor

Target variable:
ClaimNb


In [4]:
import numpy as np
import pandas as pd

In [5]:
df=pd.read_csv('Project_description_and_data/claims_train.csv')
print(df.head())
# print(df.info())

       IDpol  ClaimNb  Exposure Area  VehPower  VehAge  DrivAge  BonusMalus  \
0  2122523.0        0      0.43    D         7      18       36          95   
1  3173420.0        0      0.10    D         7      17       80          95   
2  1188619.0        0      0.33    E         7       3       36          76   
3    31400.0        0      0.56    A         5       4       73          52   
4  3138755.0        0      0.27    E         8       0       37          50   

  VehBrand   VehGas  Density Region  
0       B1  Regular     1054    R24  
1       B2  Regular      598    R25  
2       B6  Regular     4172    R82  
3      B13   Diesel       15    R24  
4      B11   Diesel     3021    R53  


In [17]:
import numpy as np
import pandas as pd
# from sklearn.model_selection import train_test_split      #it's for the not from scratch split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor


# Preprocessing
df=pd.read_csv('Project_description_and_data/claims_train.csv')

df['VehAge_log'] = np.log(df['VehAge'] + 1)
df['DrivAge_log'] = np.log(df['DrivAge'])
df['Density_log'] = np.log(df['Density'])


categorical_cols = ['VehBrand', 'VehGas', 'Area', 'Region']
for col in categorical_cols:
    unique_vals = df[col].unique()
    mapping = {val: idx for idx, val in enumerate(unique_vals)}
    df[col] = df[col].map(mapping)


features = ['Exposure','VehBrand','VehGas','VehPower',
            'VehAge_log','DrivAge_log','Area','Density_log',
            'Region','BonusMalus']
X = df[features].values
y = df['ClaimNb'].values

# Manual train/validation split
np.random.seed(42)
indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.8 * len(X))
train_idx, val_idx = indices[:split], indices[split:]

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

# # NOT FROM-SCRATCH SPLIT 
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


# Decision Tree Regressor from scratch
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
    
    def is_leaf(self):
        return self.value is not None

def mse(y):
    if len(y) == 0:
        return 0
    mean = np.mean(y)
    return np.mean((y - mean) ** 2)

def weighted_mse(y_left, y_right):
    n = len(y_left) + len(y_right)
    return (len(y_left)/n) * mse(y_left) + (len(y_right)/n) * mse(y_right)

class DecisionTreeRegressorScratch:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
    
    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)
        return self
    
    def _build_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        
        # stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split:
            leaf_value = np.mean(y)
            return Node(value=leaf_value)
        
        # find best split
        best_feature, best_threshold = self._best_split(X, y)
        if best_feature is None:
            leaf_value = np.mean(y)
            return Node(value=leaf_value)
        
        # split data
        left_indices = X[:, best_feature] <= best_threshold
        right_indices = X[:, best_feature] > best_threshold
        
        left_child = self._build_tree(X[left_indices], y[left_indices], depth+1)
        right_child = self._build_tree(X[right_indices], y[right_indices], depth+1)
        
        return Node(feature=best_feature, threshold=best_threshold,
                    left=left_child, right=right_child)
    
    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        if n_samples <= 1:
            return None, None
        
        best_mse = float('inf')
        best_feature, best_threshold = None, None
        
        for feature_idx in range(n_features):
            X_col = X[:, feature_idx]
            thresholds = np.unique(X_col)
            
            for i in range(len(thresholds) - 1):
                threshold = (thresholds[i] + thresholds[i+1]) / 2
                left_indices = X_col <= threshold
                right_indices = X_col > threshold
                
                left_y, right_y = y[left_indices], y[right_indices]
                w_mse = weighted_mse(left_y, right_y)
                
                if w_mse < best_mse:
                    best_mse = w_mse
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
    
    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)


def mean_squared_error_manual(y_true, y_pred):
    errors = (y_true - y_pred) ** 2
    return np.mean(errors)

def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res/ss_tot

# Train and evaluate
scratch_tree = DecisionTreeRegressorScratch(max_depth=7, min_samples_split=200)
scratch_tree.fit(X_train, y_train)
y_val_pred_scratch = scratch_tree.predict(X_val)

print("From-scratch Decision Tree:")
print("Validation MSE:", mean_squared_error_manual(y_val, y_val_pred_scratch))
print("Validation R²:", r2_score_manual(y_val, y_val_pred_scratch))



# Reference implementation
ref_tree = DecisionTreeRegressor(max_depth=7, min_samples_split=200, random_state=42)
ref_tree.fit(X_train, y_train)
y_val_pred_ref = ref_tree.predict(X_val)

print("\nscikit-learn Decision Tree:")
print("Validation MSE:", mean_squared_error(y_val, y_val_pred_ref))
print("Validation R²:", r2_score(y_val, y_val_pred_ref))


From-scratch Decision Tree:
Validation MSE: 0.055994660364440695
Validation R²: 0.02961455384843681

scikit-learn Decision Tree:
Validation MSE: 0.055994660364440695
Validation R²: 0.02961455384843681


In [ ]:
# Hyperparameter tuning loop
# It takes around 10min to run
depth_values = [3, 5, 7, 10]
min_samples_values = [10, 50, 200]

results = []

print("Scratch Tree Results:")
for depth in depth_values:
    for min_samples in min_samples_values:
        scratch_tree = DecisionTreeRegressorScratch(max_depth=depth, min_samples_split=min_samples)
        scratch_tree.fit(X_train, y_train)
        y_val_pred = scratch_tree.predict(X_val)
        
        mse_val = mean_squared_error(y_val, y_val_pred)
        r2_val = r2_score(y_val, y_val_pred)
        
        results.append(("scratch", depth, min_samples, mse_val, r2_val))
        print(f"depth={depth}, min_samples={min_samples} -> MSE={mse_val:.4f}, R²={r2_val:.4f}")

print("\nscikit-learn Tree Results:")
for depth in depth_values:
    for min_samples in min_samples_values:
        ref_tree = DecisionTreeRegressor(max_depth=depth, min_samples_split=min_samples, random_state=42)
        ref_tree.fit(X_train, y_train)
        y_val_pred_ref = ref_tree.predict(X_val)
        
        mse_val_ref = mean_squared_error(y_val, y_val_pred_ref)
        r2_val_ref = r2_score(y_val, y_val_pred_ref)
        
        results.append(("sklearn", depth, min_samples, mse_val_ref, r2_val_ref))
        print(f"depth={depth}, min_samples={min_samples} -> MSE={mse_val_ref:.4f}, R²={r2_val_ref:.4f}")


Scratch Tree Results:
depth=3, min_samples=10 -> MSE=0.0567, R²=0.0180
depth=3, min_samples=50 -> MSE=0.0567, R²=0.0180
depth=3, min_samples=200 -> MSE=0.0567, R²=0.0180
depth=5, min_samples=10 -> MSE=0.0562, R²=0.0257
depth=5, min_samples=50 -> MSE=0.0562, R²=0.0257
depth=5, min_samples=200 -> MSE=0.0562, R²=0.0257
depth=7, min_samples=10 -> MSE=0.0561, R²=0.0285
depth=7, min_samples=50 -> MSE=0.0560, R²=0.0297
depth=7, min_samples=200 -> MSE=0.0559, R²=0.0306
depth=10, min_samples=10 -> MSE=0.0569, R²=0.0137
depth=10, min_samples=50 -> MSE=0.0563, R²=0.0241
depth=10, min_samples=200 -> MSE=0.0560, R²=0.0296

scikit-learn Tree Results:
depth=3, min_samples=10 -> MSE=0.0567, R²=0.0180
depth=3, min_samples=50 -> MSE=0.0567, R²=0.0180
depth=3, min_samples=200 -> MSE=0.0567, R²=0.0180
depth=5, min_samples=10 -> MSE=0.0562, R²=0.0257
depth=5, min_samples=50 -> MSE=0.0562, R²=0.0257
depth=5, min_samples=200 -> MSE=0.0562, R²=0.0257
depth=7, min_samples=10 -> MSE=0.0561, R²=0.0285
depth=7, m